# Experiment 1 Under-12h Global Feature Extraction

Run this notebook on a Google Colab GPU after the local render QC/post-render step has produced `data/exp1_under12h/manifests/render_valid.parquet` and the rendered image tree. It extracts frozen global features for CLIP ViT-B/16, CLIP ViT-L/14, and DINOv2 ViT-B at the configured layers.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Update this if your repo lives elsewhere in Drive.
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/cv-project')
os.environ['CV_PROJECT_ROOT'] = str(PROJECT_ROOT)
%cd {PROJECT_ROOT}

!nvidia-smi

In [ ]:
import sys

# Install project and model dependencies. Re-running this cell is safe.
!{sys.executable} -m pip install -q -r requirements.txt
!{sys.executable} -m pip install -q transformers open_clip_torch pyarrow safetensors

In [ ]:
import pandas as pd
from pathlib import Path

CONFIG = 'configs/exp1_under12h.yaml'
SOURCE_RENDER_MANIFEST = Path('data/exp1_under12h/manifests/render_valid.parquet')
VALID_RENDER_MANIFEST = Path('data/exp1_under12h/manifests/render_valid_colab.parquet')
FEATURE_DIR = Path('data/exp1_under12h/features')
PATH_COLUMNS = ['rgb_path', 'depth_path', 'normal_path', 'mask_path']
LOCAL_PROJECT_ROOT = Path('/Users/jerry/cv-project')

if not SOURCE_RENDER_MANIFEST.is_file():
    raise FileNotFoundError(f'Missing {SOURCE_RENDER_MANIFEST}. Run local post_render first and sync data to Drive.')

manifest = pd.read_parquet(SOURCE_RENDER_MANIFEST)
for column in PATH_COLUMNS:
    if column in manifest.columns:
        def rebase_path(value):
            path = Path(str(value))
            if path.is_absolute():
                try:
                    path = path.relative_to(LOCAL_PROJECT_ROOT)
                except ValueError:
                    path = Path(*path.parts[1:])
            return str(PROJECT_ROOT / path)
        manifest[column] = manifest[column].map(rebase_path)

VALID_RENDER_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
manifest.to_parquet(VALID_RENDER_MANIFEST, index=False)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Validated renders: {len(manifest):,}')
print(f'Colab manifest: {VALID_RENDER_MANIFEST}')
print(manifest.groupby(['split', 'texture_condition']).size())

In [ ]:
# CLIP ViT-B/16 global features. All requested layers are extracted in one model pass.
!PYTHONPATH=. python scripts/extract_exp1_features.py \
  --config {CONFIG} \
  --render-manifest {VALID_RENDER_MANIFEST} \
  --feature-dir {FEATURE_DIR} \
  --models clip_vit_b16 \
  --layers final layer4 layer8 layer12 \
  --device cuda \
  --batch-size 96 \
  --num-workers 4 \
  --amp

In [ ]:
# CLIP ViT-L/14 is larger; reduce batch size if Colab reports CUDA OOM.
!PYTHONPATH=. python scripts/extract_exp1_features.py \
  --config {CONFIG} \
  --render-manifest {VALID_RENDER_MANIFEST} \
  --feature-dir {FEATURE_DIR} \
  --models clip_vit_l14 \
  --layers final layer4 layer8 layer12 \
  --device cuda \
  --batch-size 48 \
  --num-workers 4 \
  --amp

In [ ]:
# DINOv2 ViT-B global features.
!PYTHONPATH=. python scripts/extract_exp1_features.py \
  --config {CONFIG} \
  --render-manifest {VALID_RENDER_MANIFEST} \
  --feature-dir {FEATURE_DIR} \
  --models dinov2_vit_b \
  --layers final layer4 layer8 layer12 \
  --device cuda \
  --batch-size 64 \
  --num-workers 4 \
  --amp

In [ ]:
# Quick cache inventory for sync/checkpointing.
for path in sorted(FEATURE_DIR.glob('**/*')):
    if path.is_file() and path.suffix in {'.npz', '.parquet', '.pt'}:
        size_gb = path.stat().st_size / 1e9
        print(f'{path}  {size_gb:.2f} GB')